In [1]:
import numpy as np

## 1 Training

### 1.1 Training Algorithm

### 1.2 Implementation

### 1.3 Data Sensitivity

#### Impact of Outliers

- Decision trees are generally robust to outliers
- Decision trees can be sensitive to extreme values in the target variable.

#### Impact of Varied Scales

- Tree based model do not require Feature Scaling because optimization is not based on Gradient based algorithms.
- Decisions are not influenced by the scale of a particular feature.

#### Need for Feature Encoding

- Decision Tree require Feature Encoding.
- Avoid One-Hot encoding of features as it increases the complexity of tree's feature-search space. 

#### Impact of Imbalanced Dataset

- When using Decision trees for Classification problems they are impacted by imbalanced dataset.
- This is because Gini impurity and Entropy naturally favor the majority class.

## 2 Bias Variance Tradeoff

### 2.2 Underfitting

#### Scenarios for Overfitting

- A shallow Tree

#### How to detect underfitting?

When the Training Error is high or Training Accuracy is very low.

### 2.3 Overfitting

#### What is Pruning?

- Pruning in a decision tree is a technique that removes unnecessary branches (splits) or nodes to avoid overfitting.
- Pruning is a Depth control mechanism applied on Decision Tree achieved using Hyperparameters.

#### Scenarios for Overfitting

1. Too much perfection i.e., splitting until every Leaf node is perfectly pure.
2. Very less number of data-points in any of the leaf nodes.
3. Overall in a Tree, there are unnecessarily large number of leaf nodes (may or may not be perfectly pure).
4. Decision Tree with too many levels or depth.

### 2.4 Hyperparameters

#### 1 Minimum Samples Split

- This hyperparameter is used in Pruning a Decision Tree.
- Minimum number of data-points that must be present in a node before qualifying for a spit.
- This hyperparameter is applied before splitting.

#### 2 Minimum Samples Leaf

- This hyperparameter is used in Pruning a Decision Tree.
- The minimum number of data-points, say $s$, required to form a leaf node.
- If any of the child nodes resulting from a split has less than $s$ number of data-points, then the split operation on parent node is aborted.
- This hyperparameter is applied after splitting.

#### 3 Max Leaf Nodes

- This hyperparameter is used in Pruning a Decision Tree.
- Maximum number of leaf nodes allowed in the entire Tree structure.
- Total number of Leaf Nodes remaining after applying this hyperparameter may or may not be perfectly pure.

#### 4 Max Depth

- This hyperparameter is used in Pruning and is most important in avoiding overfitting in a Decision Tree.
- Maximum depth allowed in the Tree structure.

#### 5 Max Features

- This hyperparameter is used to reduce the Time complexity of Training process.
- The number of features `d` to consider when looking for the best split.
- For example, when `d = 1` only one feature is considered at a time while looking for best split.

## 3 Feature Importance

### 3.1 FI Calculation

#### How Feature Importance is calculated?

Feature Importance is calculated using the Normalized total Gain (Information Gain or Gini Gain) contributed by the feature.

#### Formula

### 3.2 Implementation

#### Function to calculate Gini Gain

In [2]:
def gini_impurity(splits: list):
    """
    Function to calculate Gini Impurity using Formula #2.
    """
    probs = np.asarray(splits) / np.sum(splits)
    gi = 0
    for p_i in probs.tolist():
        gi += p_i * (1 - p_i)

    return gi


def gini_gain(parent: list, splits: list[list]):
    """
    Function to calculate Gini Gain.
    """
    # Calculate Gini-Impurity of parent.
    g_parent = gini_impurity(parent)
    s_tot = sum(parent)

    # Calculate Gini weighted average of splits.
    g_split = 0
    for child in splits:
        # Calculate Gini-Impurity of child.
        g_child = gini_impurity(child)
        si_tot = sum(child)
        g_split += si_tot / s_tot * g_child

    # Calculate Gini-Gain.
    gini_gain = g_parent - g_split

    return g_parent, g_split, gini_gain

#### Function to compute Feature Importance

In [3]:
def feature_importance_v1(tree):
    """
    Function to calculate Feature importance based on a Decision Tree.
    """
    ftr_gains = dict()
    tot_gains = 0
    for nodes in tree:
        _, _, g_gain = gini_gain(parent=nodes["parent"], splits=nodes["splits"])
        ftr_gains[nodes["feature"]] = ftr_gains.get(nodes["feature"], 0) + g_gain
        tot_gains += g_gain

    # Normalize Gains to calculate feature importance.
    ftr_imp = {ftr: round(gain / tot_gains, 2) for ftr, gain in ftr_gains.items()}

    return ftr_imp

In [4]:
def feature_importance_v2(tree):
    """
    Function to calculate Feature importance based on a Decision Tree.
    """
    ftr_gains = dict()
    tot_gains = 0
    for nodes in tree:
        tot_samples = sum(nodes["parent"])
        _, _, g_gain = gini_gain(parent=nodes["parent"], splits=nodes["splits"])
        w_gain = tot_samples * g_gain  # Weighted Gain
        ftr_gains[nodes["feature"]] = ftr_gains.get(nodes["feature"], 0) + w_gain
        tot_gains += w_gain

    # Normalize Gains to calculate feature importance.
    ftr_imp = {ftr: round(gain / tot_gains, 2) for ftr, gain in ftr_gains.items()}

    return ftr_imp

### 3.3 Examples

#### Example #1

Calculate Feature importance of below tree.

![Sample Decision Tree](images/2_emp_att_dt.png)

In [5]:
dsn_tree = [
    # Level 1:
    {"feature": "Age", "parent": [42, 28], "splits": [[30, 27], [12, 1]]},
    # Level 2:
    {"feature": "Age", "parent": [30, 27], "splits": [[26, 15], [4, 12]]},
    # Level 3:
    {"feature": "Ot_Hrs", "parent": [26, 15], "splits": [[21, 5], [5, 10]]},
    # Level 3:
    {"feature": "Ot_Hrs", "parent": [4, 12], "splits": [[0, 12], [4, 0]]},
]

In [6]:
feature_importance_v1(dsn_tree)

{'Age': 0.18, 'Ot_Hrs': 0.82}

In [7]:
feature_importance_v2(dsn_tree)

{'Age': 0.4, 'Ot_Hrs': 0.6}

#### Example #2

In [8]:
dsn_tree = [
    # Level 1:
    {"feature": "Age", "parent": [4, 11], "splits": [[3, 3], [1, 8]]},
    # Level 2:
    {"feature": "Ot_Hrs", "parent": [3, 3], "splits": [[1, 0], [2, 3]]},
    # Level 3:
    {"feature": "Ot_Hrs", "parent": [2, 3], "splits": [[0, 3], [2, 0]]},
]

In [9]:
feature_importance_v1(dsn_tree)

{'Age': 0.11, 'Ot_Hrs': 0.89}

In [10]:
feature_importance_v2(dsn_tree)

{'Age': 0.27, 'Ot_Hrs': 0.73}